In [1]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

np.random.seed(0)

# Problem data
h = 0.01
Q = 1
R = 1.5**2

A = np.array([[1, h], [0, 1]])
G = np.array([[0], [1]])
C = np.array([[1, 0]])

x0_tilde = np.array([[0], [10]])
P0 = np.array([[100, 0], [0, 1]])

In [2]:
# Measurement and Time update functions

def measurement_update(sigma_tu, x_tu, y):
    z = C @ sigma_tu @ C.T + R  # auxiliary variable
    x_mu = x_tu + sigma_tu @ C.T @ np.linalg.solve(z, y - C@x_tu)
    sigma_mu = sigma_tu - sigma_tu @ C.T @ np.linalg.solve(z, C@sigma_tu)
    return sigma_mu, x_mu

def time_update(sigma_mu, x_mu):
    x_tu = A @ x_mu
    sigma_tu = A @ sigma_mu @ A.T + Q * (G @ G.T)
    return sigma_tu, x_tu

In [ ]:
# Implementation of KF
n_sim = 5000
x = np.random.multivariate_normal(x0_tilde.flatten(), P0, 1).reshape((-1,1))  # random initial state
sigma_tu, x_tu = P0, x0_tilde.reshape((-1, 1))  # initialisation

x_cache = np.zeros((n_sim, 2))
x_mu_cache = np.zeros((n_sim-1, 2))
sigma_mu_cache = np.zeros((n_sim-1, 2, 2))
x_cache[0, :] = x.T

for t in range(n_sim - 1):
    v = np.random.normal(0, np.sqrt(R), 1)
    y = C @ x + v
    sigma_mu, x_mu = measurement_update(sigma_tu, x_tu, y)
    x_mu_cache[t, :] = x_mu.T
    sigma_mu_cache[t] = sigma_mu
    sigma_tu, x_tu = time_update(sigma_mu, x_mu)
    w = np.random.normal(0, np.sqrt(Q), 1).T
    x = A @ x + G * w
    x_cache[t+1, :] = x.T

t_axis = np.arange(n_sim) * h